# Experiment: Inspect a bounded AAPL ITCH replay sample

**Objective:** locate AAPL in a NASDAQ ITCH feed and decode a small, representative set of order-book messages into Polars.

**Success criteria:** the notebook runs top-to-bottom, collects exactly `MAX_ROWS` AAPL messages, and summarizes their message types without scanning or printing the entire feed.

## Plan

1. Resolve the feed from `LOBO_ITCH_PATH` or the repository's default data path.
2. Read Stock Directory messages until AAPL's `stock_locate` is found.
3. Decode at most `MAX_ROWS` add, execute, cancel, replace, and delete messages.
4. Inspect a small preview and message-type counts.

In [ ]:
from dataclasses import asdict
import os
from pathlib import Path

import polars as pl
from itch.parser import MessageParser

MAX_ROWS = 500
configured_path = os.environ.get("LOBO_ITCH_PATH")
if configured_path:
    DATA_PATH = Path(configured_path).expanduser().resolve()
else:
    candidates = (
        Path("data/NASDAQ/01302020.NASDAQ_ITCH50"),
        Path("../data/NASDAQ/01302020.NASDAQ_ITCH50"),
    )
    DATA_PATH = next((path.resolve() for path in candidates if path.is_file()), None)

if DATA_PATH is None or not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Set LOBO_ITCH_PATH to a NASDAQ ITCH 5.0 feed or place the fixture "
        "at data/NASDAQ/01302020.NASDAQ_ITCH50."
    )

DATA_PATH

## Locate AAPL

ITCH order messages carry a compact numeric `stock_locate`. The Stock Directory record provides the mapping from that value to the ticker.

In [ ]:
directory_parser = MessageParser(message_type=b"R")
aapl_locate = None

with DATA_PATH.open("rb") as feed:
    for message in directory_parser.parse_file(feed):
        if message.stock.rstrip() == b"AAPL":
            aapl_locate = message.stock_locate
            break

assert aapl_locate is not None, "AAPL was not present in the Stock Directory records"
aapl_locate

## Decode a bounded sample

Filtering by `stock_locate` happens before decoding. The loop stops as soon as the requested sample is complete, keeping execution quick even when the source feed is very large.

In [ ]:
book_parser = MessageParser(message_type=b"AFECXDU")
rows = []

with DATA_PATH.open("rb") as feed:
    for message in book_parser.parse_file(feed):
        if message.stock_locate != aapl_locate:
            continue
        rows.append(asdict(message.decode()))
        if len(rows) == MAX_ROWS:
            break

assert len(rows) == MAX_ROWS, f"expected {MAX_ROWS} AAPL messages, found {len(rows)}"
messages = pl.from_dicts(rows, strict=False)
messages.head(10)

In [ ]:
message_counts = (
    messages.group_by(["message_type", "description"])
    .len(name="messages")
    .sort("messages", descending=True)
)

assert message_counts["messages"].sum() == MAX_ROWS
message_counts

## Notes and next steps

- The sample is intentionally bounded; increasing `MAX_ROWS` changes runtime and memory use.
- A missing feed produces an actionable error instead of relying on hidden notebook state.
- For a full performance replay, use the Rust ITCH examples and benchmarks rather than accumulating every decoded message in Python.

**Next experiment:** group executions by price or timestamp window and compare them with the reconstructed Rust book.